In [1]:
from modeling_distillemb import BertModel, BertForSequenceClassification, BertForTokenClassification
from distill_emb import DistillEmbSmall, DistillEmb
from config import DistillModelConfig, DistillEmbConfig
import torch
from transformers import AutoTokenizer, RwkvConfig, RwkvModel, AutoModel
from tokenizer import CharTokenizer
from knn_classifier import KNNTextClassifier
from data_loader import load_sentiment, load_ner_dataset, load_pos_dataset
from data_loader import load_news_dataset
import pandas as pd
from retrieval import build_json_pairs, top1_accuracy
import os
from transformers import GPT2LMHeadModel
from data_loader import *
from datasets import Dataset, DatasetDict

In [2]:
num_input_chars=12

In [3]:
tokenizer = CharTokenizer.from_pretrained(pretrained_directory="distil-emb-base")
distill_config = DistillEmbConfig.from_pretrained(pretrained_model_name_or_path="distil-emb-base")
distill_model = DistillEmb.from_pretrained(pretrained_model_name_or_path="distil-emb-base")

In [4]:
# distill_config.distill_dropout = 0.25
config = DistillModelConfig(
    vocab_size=30522,
    hidden_size=1024,
    num_hidden_layers=3,
    num_attention_heads=8,
    intermediate_size=3072,
    max_position_embeddings=1024,
    type_vocab_size=2,
    pad_token_id=0,
    position_embedding_type="absolute",
    use_cache=True,
    classifier_dropout=None,
    hidden_dropout_prob=0.1,
    embedding_type="distill",  # 'distilemb', 'fasttext'
    encoder_type='lstm', #'lstm'
    num_input_chars=num_input_chars,  # number of characters in each token
    char_vocab_size=tokenizer.char_vocab_size,
    distill_config=distill_config,
    distill_pretrained_model_name="distil-emb-base",
    is_decoder=False
)


In [5]:
df, labels = load_pos_dataset()
labels = list(range(max(labels) + 1))

df['text'] = df['tokens'].apply(lambda x: ' '.join(x))
# remove empty text rows
df = df[df['text'].str.strip().astype(bool)].sample(frac=1.0, random_state=42).reset_index(drop=True)

Loaded 30494 rows from masakhapos.parquet columns Index(['id', 'tokens', 'labels', 'lang', 'split'], dtype='object')


In [6]:
df

,id,tokens,labels,lang,split,text
0,151,"[Tògán, Patrice, TALƆN, ɔ́, "", yí, afɔ, sɔ, ɖo...","[0, 10, 10, 8, 1, 16, 0, 16, 2, 16, 6, 0, 2, 0...",fon,train,"Tògán Patrice TALƆN ɔ́ "" yí afɔ sɔ ɖo tè tɔ tò..."
1,709,"[Vʋʋsma, loogr, poorẽ, ivoaryẽma, lebsa, zẽmta...","[0, 0, 2, 10, 16, 0, 0, 3, 2, 7, 10, 10, 0, 6,...",mos,train,Vʋʋsma loogr poorẽ ivoaryẽma lebsa zẽmtaar min...
2,379,"[president, Emmanuel, Macron, of, France, don,...","[0, 10, 10, 2, 10, 17, 16, 8, 16, 2, 10, 10, 1...",pcm,train,president Emmanuel Macron of France don suspen...
3,137,"[Perezida, Jovenel, wayoboraga, Haïti, kuva, m...","[0, 10, 16, 10, 16, 2, 3, 16, 2, 0, 11, 16, 1]",kin,train,Perezida Jovenel wayoboraga Haïti kuva mu 2016...
4,247,"[A, bɛ, hakilijigin, kɛ, k', a, y', a, fɔ, kab...","[11, 17, 0, 16, 7, 11, 7, 11, 16, 0, 0, 3, 1, ...",bam,test,A bɛ hakilijigin kɛ k' a y' a fɔ kabini san sa...
...,...,...,...,...,...,...
30489,99,"[Ka, wano, to, mu, ,, na, nka, hwee, .]","[16, 16, 11, 2, 1, 9, 16, 16, 1]",twi,dev,"Ka wano to mu , na nka hwee ."
30490,14,"[Erias, Lukwago, ye, mubaka, wa, Paalamenti, o...","[10, 10, 17, 0, 2, 0, 2, 10, 6, 1]",lug,train,Erias Lukwago ye mubaka wa Paalamenti owa Kamp...
30491,85,"[Daʼgaə́, mthə́dzə, ntʉ́m, gɔ̂pnaʼ, pə́, wə́, ...","[9, 0, 2, 0, 17, 17, 16, 0, 7, 16, 10, 7, 16, ...",bbj,train,Daʼgaə́ mthə́dzə ntʉ́m gɔ̂pnaʼ pə́ wə́ jɔ́ mjy...
30492,532,"[ƐFIYƐSIDE, ,, hali, n', a, waatilaɲɛmɔgɔya, t...","[0, 1, 0, 5, 11, 0, 7, 17, 10, 10, 10, 0, 1, 0...",bam,test,"ƐFIYƐSIDE , hali n' a waatilaɲɛmɔgɔya tun bɛ S..."


In [7]:
lang_counts = df.groupby('split')['lang'].nunique()
for split, count in lang_counts.items():
    print(f"{split.capitalize()} split has {count} languages.")

Dev split has 20 languages.
Test split has 20 languages.
Train split has 20 languages.


In [8]:
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
config.label2id = label2id
config.id2label = id2label

print(f"Converted labels to integers: {label2id}")
print(f"Converted integers to labels: {id2label}")

Converted labels to integers: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17}
Converted integers to labels: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17}


In [9]:
config.num_labels = len(label2id)
model = BertForTokenClassification(config)

In [10]:
model.save_pretrained("ner-model")
tokenizer.save_pretrained("ner-model")

In [12]:
model.from_pretrained("ner-model")

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): DistilEmbeddings(
      (word_embeddings): DistillEmb(
        (encoder): DistillEmbBase(
          (embedding): Embedding(1518, 128)
          (conv1): Conv1d(12, 128, kernel_size=(5,), stride=(1,))
          (conv2): Conv1d(128, 256, kernel_size=(5,), stride=(1,))
          (conv3): Conv1d(256, 384, kernel_size=(5,), stride=(1,))
          (conv4): Conv1d(384, 448, kernel_size=(3,), stride=(1,))
          (conv5): Conv1d(448, 512, kernel_size=(3,), stride=(1,))
          (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (output_layer): Linear(in_features=512, out_features=512, bias=True)
          (activation): GELU(approximate='none')
          (norm0): LayerNorm((12, 128), eps=1e-05, elementwise_affine=True)
          (norm1): LayerNorm((128, 62), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256, 29), eps=1e-05, elementwise_affine=True)
          (norm3

In [ ]:

train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

In [ ]:

# Create HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset

In [ ]:
from typing import Dict, Any

def preprocess_function(examples: Dict[str, Any]):
    batch = tokenizer(
        examples["text"],
        padding=False,
        max_length=512,
        return_attention_mask=False,
    )

    batch["labels"] = examples["labels"]
    return batch

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding="longest",
            max_length=512,
            return_tensors="pt",
            return_attention_mask=True,
        )
        
        max_len = batch["input_ids"].shape[1] - 2  # exclude special tokens
        padded_labels = []
        for f in features:
            label = f["labels"]
            
            padded_label = [-100] +  label + [-100] * (max_len - len(label)) + [-100]
            padded_labels.append(padded_label)
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        
        assert batch["labels"].shape == (batch["input_ids"].shape[0], batch["input_ids"].shape[1]), f"Labels shape {batch['labels'].shape} does not match input_ids shape {batch['input_ids'].shape}"
        return batch

data_collator = CustomDataCollator(tokenizer)

In [ ]:
##### from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    pred_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        mask = label_seq != -100
        true_labels.extend(label_seq[mask])
        pred_labels.extend(pred_seq[mask])

    label_ids = list(label2id.values())

    return {
        "accuracy": accuracy_score(true_labels, pred_labels),
        "f1_weighted": f1_score(true_labels, pred_labels, average="weighted", labels=label_ids, zero_division=0),
        "f1_macro": f1_score(true_labels, pred_labels, average="macro", labels=label_ids, zero_division=0),
        "f1_micro": f1_score(true_labels, pred_labels, average="micro", labels=label_ids, zero_division=0),
    }


import os
dataloader_num_workers=os.cpu_count() - 1
batch_size = 16

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=20,
    weight_decay=0.0,
    report_to=[],
    eval_strategy="epoch",  
    save_total_limit=1,
    save_only_model=True,
    logging_strategy="steps",
    logging_steps=10,
    label_smoothing_factor=0.0,
    max_grad_norm=5.0,
    warmup_ratio=0.0,
    lr_scheduler_type="cosine",
    dataloader_num_workers=1,        # Number of CPU workers for data loading
    dataloader_pin_memory=True,      # Faster GPU transfer
    gradient_accumulation_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate the model after training
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
labels

In [ ]:
from collections import defaultdict

pred_output = trainer.predict(tokenized_test)
logits = pred_output.predictions
label_ids = pred_output.label_ids
langs = test_df["lang"].tolist()

lang_true, lang_pred = defaultdict(list), defaultdict(list)
for idx, lang in enumerate(langs):
    label_seq = label_ids[idx]
    pred_seq = logits[idx].argmax(axis=-1)
    mask = label_seq != -100
    if not np.any(mask):
        continue
    lang_true[lang].extend(label_seq[mask])
    lang_pred[lang].extend(pred_seq[mask])

label_id_list = list(label2id.values())
lang_metrics = {}
for lang, true_values in lang_true.items():
    preds = lang_pred[lang]
    lang_metrics[lang] = {
        "accuracy": accuracy_score(true_values, preds),
        "f1_weighted": f1_score(true_values, preds, average="weighted", labels=label_id_list, zero_division=0),
        "f1_macro": f1_score(true_values, preds, average="macro", labels=label_id_list, zero_division=0),
        "f1_micro": f1_score(true_values, preds, average="micro", labels=label_id_list, zero_division=0),
        "num_tokens": len(true_values),
    }

lang_metrics_df = pd.DataFrame.from_dict(lang_metrics, orient="index").sort_values("f1_macro", ascending=False)
lang_metrics_df